In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split

# Load your dataset
# Assuming your dataset is in a CSV file with columns: 'Sentence', 'Label 1', 'Label 2', 'Label 3'
data = pd.read_csv('type_classification.csv')

# Split the dataset into training and validation sets
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

# Define a custom dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Define the maximum length of the input sequences
MAX_LEN = 128

# Create instances of the dataset
train_dataset = TextDataset(
    texts=train_data['sentence'].to_numpy(),
    labels=train_data[['structure_focus','process_focus', 'usecase_focus']].values,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_data['sentence'].to_numpy(),
    labels=val_data[['structure_focus','process_focus', 'usecase_focus']].values,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
BATCH_SIZE = 16

train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_data_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Define the model
class BERTLabelAttention(nn.Module):
    def __init__(self, bert_model_name, num_labels):
        super(BERTLabelAttention, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.3)
        self.label_attention = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.out = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False
        )
        pooled_output = self.dropout(pooled_output)
        attention_scores = self.label_attention(pooled_output)
        attention_probs = torch.softmax(attention_scores, dim=-1)
        context_vector = torch.bmm(attention_probs.unsqueeze(1), pooled_output.unsqueeze(2)).squeeze(2)
        output = self.out(context_vector)
        return output

# Initialize the model, loss function, and optimizer
model = BERTLabelAttention(bert_model_name='bert-base-uncased', num_labels=3)
model = model.cuda()

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

# Training loop
def train_epoch(model, data_loader, criterion, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0

    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    return sum(losses) / len(losses)

# Evaluation loop
def eval_model(model, data_loader, criterion, device, n_examples):
    model = model.eval()
    losses = []
    preds = []
    true_labels = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs, labels)
            losses.append(loss.item())

            preds.append(outputs.detach().cpu())
            true_labels.append(labels.detach().cpu())

    preds = torch.cat(preds).numpy()
    true_labels = torch.cat(true_labels).numpy()

    preds = (preds > 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, preds, average='macro')

    return sum(losses) / len(losses), precision, recall, f1

# Training and evaluation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 3

total_steps = len(train_data_loader) * EPOCHS

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)

    train_loss = train_epoch(
        model,
        train_data_loader,
        criterion,
        optimizer,
        device,
        scheduler,
        len(train_data)
    )

    print(f'Train loss {train_loss}')

    val_loss, val_precision, val_recall, val_f1 = eval_model(
        model,
        val_data_loader,
        criterion,
        device,
        len(val_data)
    )

    print(f'Val Loss {val_loss} Val Precision {val_precision} Val Recall {val_recall} Val F1 Score {val_f1}')

# Prediction function
def predict(model, sentence, tokenizer, max_len, device):
    model = model.eval()
    encoding = tokenizer.encode_plus(
        sentence,
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    outputs = torch.sigmoid(outputs)
    return (outputs > 0.5).int().tolist()

# Example prediction
example_sentence = "Your example sentence here"
predicted_labels = predict(model, example_sentence, tokenizer, MAX_LEN, device)
print(f'Predicted labels: {predicted_labels}')


C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.9\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.9\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_

AssertionError: Torch not compiled with CUDA enabled